# AWS SageMaker — Deploying a Model to a Managed Endpoint

AWS SageMaker is Amazon's managed ML platform. It handles the infrastructure for training, hosting, and monitoring your models so you don't have to manage EC2 instances or load balancers yourself.

This notebook walks through every step of the SageMaker deployment flow using real boto3 API calls. Since you likely don't have AWS credentials in this environment, every call is wrapped in `try/except` to catch authentication errors and print what the call would do in production.

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain the SageMaker deployment flow from local model to live endpoint
2. Create the `model.tar.gz` artifact format SageMaker expects
3. Read and understand the boto3 calls for `create_model`, `create_endpoint_config`, and `create_endpoint`
4. Invoke a SageMaker endpoint and interpret the response
5. Set up auto-scaling and delete an endpoint to stop billing

## 1. SageMaker Architecture

SageMaker has three layers that work together:

```
Your model artifact (S3)
      |
      v
SageMaker Model  <-- points to S3 path + inference container image
      |
      v
Endpoint Config  <-- defines instance type, count, traffic splits
      |
      v
Endpoint         <-- the live HTTPS URL your application calls
```

Separating the **Model** from the **Endpoint Config** from the **Endpoint** lets you:
- Swap a new model version without touching the config
- Run A/B tests by routing traffic to two model variants in one endpoint config
- Rollback by repointing the endpoint to a previous config

## 2. Train a Model and Create the SageMaker Artifact

In [1]:
# WHAT: train the iris model locally and stage model.joblib + scaler.joblib.
# WHY: SageMaker separates training from hosting — we play the training side
# here, producing the files the hosted endpoint will later load.
import os
import tarfile
import pathlib
import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train a simple classifier
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_s, y_train)

print(f"Model trained. Test accuracy: {clf.score(scaler.transform(X_test), y_test):.2%}")

# Save model artifacts to a staging directory
staging_dir = pathlib.Path('/tmp/sagemaker_staging')
staging_dir.mkdir(exist_ok=True)

joblib.dump(clf, staging_dir / 'model.joblib')
joblib.dump(scaler, staging_dir / 'scaler.joblib')
print(f"Saved model.joblib and scaler.joblib to {staging_dir}")

Model trained. Test accuracy: 100.00%
Saved model.joblib and scaler.joblib to /tmp/sagemaker_staging


In [2]:
# SageMaker expects a tar.gz archive named model.tar.gz
# Inside it, the inference container looks for model artifacts at a known path

ARTIFACT_PATH = '/tmp/model.tar.gz'

with tarfile.open(ARTIFACT_PATH, 'w:gz') as tar:
    tar.add(staging_dir / 'model.joblib', arcname='model.joblib')
    tar.add(staging_dir / 'scaler.joblib', arcname='scaler.joblib')

size_kb = os.path.getsize(ARTIFACT_PATH) / 1024
print(f"Created: {ARTIFACT_PATH}  ({size_kb:.1f} KB)")

# Verify the archive contents
with tarfile.open(ARTIFACT_PATH, 'r:gz') as tar:
    print("Archive contents:")
    for member in tar.getmembers():
        print(f"  {member.name:30s} {member.size:,} bytes")

Created: /tmp/model.tar.gz  (22.7 KB)
Archive contents:
  model.joblib                   186,929 bytes
  scaler.joblib                  679 bytes


## 3. Upload the Artifact to S3

SageMaker pulls the model artifact from S3 when it starts the endpoint. The `s3_model_uri` is what you give to `create_model()` later.

In [3]:
# WHAT: upload the model.tar.gz to S3 — or, without AWS credentials, show exactly
# what would be uploaded and where.
# WHY: SageMaker can only deploy from S3; the try/except pattern lets this lesson
# run for every student while teaching the real call and its failure modes.
import boto3
from botocore.exceptions import NoCredentialsError, ClientError

BUCKET_NAME = 'my-ml-models-bucket-replace-me'
S3_KEY = 'iris-classifier/v1/model.tar.gz'

try:
    s3 = boto3.client('s3', region_name='us-east-1')
    s3.upload_file(ARTIFACT_PATH, BUCKET_NAME, S3_KEY)
    s3_model_uri = f's3://{BUCKET_NAME}/{S3_KEY}'
    print(f"Uploaded to: {s3_model_uri}")

# No AWS account in class: keep the URI so later cells can still build on it.
except NoCredentialsError:
    s3_model_uri = f's3://{BUCKET_NAME}/{S3_KEY}'  # set for later cells
    print("[No credentials] In production, set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY.")
    print(f"The upload command would put the artifact at: {s3_model_uri}")

# Real AWS rejections (bad bucket, permissions) surface with their error code.
except ClientError as e:
    print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")
    s3_model_uri = f's3://{BUCKET_NAME}/{S3_KEY}'

[No credentials] In production, set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY.
The upload command would put the artifact at: s3://my-ml-models-bucket-replace-me/iris-classifier/v1/model.tar.gz


## 4. Create the SageMaker Model

A SageMaker **Model** object registers:
- Where the artifact lives (S3 URI)
- Which Docker container image to use for inference
- The IAM role that the endpoint can use to access S3

In [4]:
# WHAT: register the artifact as a SageMaker Model — container image + S3 data + IAM role.
# WHY: a 'Model' in SageMaker is the pairing of YOUR weights with AWS's serving
# container; the ECR image URI is how you pick the sklearn runtime.
import json

# SageMaker built-in containers are identified by ECR image URIs
# This is the sklearn inference container for us-east-1
SKLEARN_IMAGE_URI = '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3'
EXECUTION_ROLE_ARN = 'arn:aws:iam::123456789012:role/SageMakerExecutionRole'
MODEL_NAME = 'iris-classifier-v1'

# The three essentials: which container, which artifact, which permissions.
create_model_params = {
    'ModelName': MODEL_NAME,
    'PrimaryContainer': {
        'Image': SKLEARN_IMAGE_URI,
        'ModelDataUrl': s3_model_uri,
        'Environment': {
            'SAGEMAKER_PROGRAM': 'inference.py',
            'SAGEMAKER_SUBMIT_DIRECTORY': s3_model_uri,
        }
    },
    'ExecutionRoleArn': EXECUTION_ROLE_ARN,
}

try:
    sm = boto3.client('sagemaker', region_name='us-east-1')
    response = sm.create_model(**create_model_params)
    print(f"Model created: {response['ModelArn']}")

# Without credentials we print the exact API payload instead of sending it.
except NoCredentialsError:
    print("[No credentials] Would call: sagemaker.create_model()")
    print("Parameters that would be sent:")
    print(json.dumps(create_model_params, indent=2))

except ClientError as e:
    print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")

[No credentials] Would call: sagemaker.create_model()
Parameters that would be sent:
{
  "ModelName": "iris-classifier-v1",
  "PrimaryContainer": {
    "Image": "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3",
    "ModelDataUrl": "s3://my-ml-models-bucket-replace-me/iris-classifier/v1/model.tar.gz",
    "Environment": {
      "SAGEMAKER_PROGRAM": "inference.py",
      "SAGEMAKER_SUBMIT_DIRECTORY": "s3://my-ml-models-bucket-replace-me/iris-classifier/v1/model.tar.gz"
    }
  },
  "ExecutionRoleArn": "arn:aws:iam::123456789012:role/SageMakerExecutionRole"
}


## 5. Create an Endpoint Configuration

The endpoint config specifies the instance type and how many instances to run. It is separate from the endpoint so you can update the config independently and then apply it in a blue/green deployment.

In [5]:
# WHAT: create the endpoint CONFIG — instance type, count, and traffic weight.
# WHY: config is separate from the endpoint on purpose: blue/green and A/B
# deployments work by swapping configs under the same endpoint name.
ENDPOINT_CONFIG_NAME = 'iris-classifier-config-v1'

endpoint_config_params = {
    'EndpointConfigName': ENDPOINT_CONFIG_NAME,
    'ProductionVariants': [
        {
            'VariantName': 'primary',
            'ModelName': MODEL_NAME,
            'InstanceType': 'ml.m5.large',   # 2 vCPU, 8 GB RAM, ~$0.115/hr
            'InitialInstanceCount': 1,
            'InitialVariantWeight': 1.0,
        }
    ]
}

try:
    sm = boto3.client('sagemaker', region_name='us-east-1')
    response = sm.create_endpoint_config(**endpoint_config_params)
    print(f"Endpoint config created: {response['EndpointConfigArn']}")

# Teaching fallback: show the parameters the call would send.
except NoCredentialsError:
    print("[No credentials] Would call: sagemaker.create_endpoint_config()")
    print("Parameters that would be sent:")
    print(json.dumps(endpoint_config_params, indent=2))

except ClientError as e:
    print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")

[No credentials] Would call: sagemaker.create_endpoint_config()
Parameters that would be sent:
{
  "EndpointConfigName": "iris-classifier-config-v1",
  "ProductionVariants": [
    {
      "VariantName": "primary",
      "ModelName": "iris-classifier-v1",
      "InstanceType": "ml.m5.large",
      "InitialInstanceCount": 1,
      "InitialVariantWeight": 1.0
    }
  ]
}


## 6. Deploy the Endpoint

Creating the endpoint provisions the infrastructure. This takes 3–10 minutes in production. The endpoint status moves from `Creating` → `InService`.

In [6]:
# WHAT: create the endpoint itself and wait until it reports InService.
# WHY: this is the step that provisions billable instances — from here on the
# model has an HTTPS URL and you are paying by the hour.
ENDPOINT_NAME = 'iris-classifier-endpoint'

create_endpoint_params = {
    'EndpointName': ENDPOINT_NAME,
    'EndpointConfigName': ENDPOINT_CONFIG_NAME,
}

try:
    sm = boto3.client('sagemaker', region_name='us-east-1')
    response = sm.create_endpoint(**create_endpoint_params)
    print(f"Endpoint creation started: {response['EndpointArn']}")
    print("Polling for status...")

    # In production: wait for endpoint to be InService
    # Endpoints take minutes to start; the waiter polls status until ready.
    waiter = sm.get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=ENDPOINT_NAME)
    print("Endpoint is InService!")

# Teaching fallback: show the call plus how status polling would look.
except NoCredentialsError:
    print("[No credentials] Would call: sagemaker.create_endpoint()")
    print("Parameters:", json.dumps(create_endpoint_params, indent=2))
    print()
    print("Then poll status with:")
    print("  sm.describe_endpoint(EndpointName='iris-classifier-endpoint')")
    print("  # Returns {'EndpointStatus': 'Creating' -> 'InService'}")

except ClientError as e:
    print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")

[No credentials] Would call: sagemaker.create_endpoint()
Parameters: {
  "EndpointName": "iris-classifier-endpoint",
  "EndpointConfigName": "iris-classifier-config-v1"
}

Then poll status with:
  sm.describe_endpoint(EndpointName='iris-classifier-endpoint')
  # Returns {'EndpointStatus': 'Creating' -> 'InService'}


## 7. Invoke the Endpoint

Once the endpoint is `InService`, any application can call it using `sagemaker-runtime`. The request body format depends on what the inference script expects — here we use CSV.

In [7]:
# WHAT: invoke the endpoint with one CSV row — or simulate the response locally.
# WHY: invoke_endpoint is the production request path; the local simulation
# proves the same model produces the answer the endpoint would return.
import io

# Sample input: one iris flower [sepal_length, sepal_width, petal_length, petal_width]
sample_input = '5.1,3.5,1.4,0.2'

invoke_params = {
    'EndpointName': ENDPOINT_NAME,
    'ContentType': 'text/csv',
    'Body': sample_input.encode('utf-8'),
}

try:
    runtime = boto3.client('sagemaker-runtime', region_name='us-east-1')
    response = runtime.invoke_endpoint(**invoke_params)
    result = response['Body'].read().decode('utf-8')
    print(f"Prediction: {result}")

# No endpoint available: run the identical prediction locally so students
# still see a real response shape.
except NoCredentialsError:
    print("[No credentials] Would call: runtime.invoke_endpoint()")
    print(f"Input sent  : {sample_input}")
    print("Response format: {'Body': StreamingBody, 'ContentType': 'text/csv', ...}")
    print()
    # Show what the response looks like by running locally
    sample = np.array([[5.1, 3.5, 1.4, 0.2]])
    sample_s = scaler.transform(sample)
    pred = clf.predict(sample_s)[0]
    prob = clf.predict_proba(sample_s)[0].max()
    simulated_response = {'prediction': iris.target_names[pred], 'confidence': round(float(prob), 3)}
    print(f"Simulated response: {simulated_response}")

except ClientError as e:
    print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")

[No credentials] Would call: runtime.invoke_endpoint()
Input sent  : 5.1,3.5,1.4,0.2
Response format: {'Body': StreamingBody, 'ContentType': 'text/csv', ...}

Simulated response: {'prediction': np.str_('setosa'), 'confidence': 1.0}


## 8. Auto-Scaling Policy

Auto-scaling lets SageMaker add or remove instances based on traffic. The most common trigger is `SageMakerVariantInvocationsPerInstance` — average requests per instance per minute.

In [8]:
# WHAT: attach auto-scaling to the endpoint: 1-10 instances tracking a target of
# 1000 invocations/min per instance.
# WHY: managed autoscaling is a big reason to use SageMaker — note the asymmetric
# cooldowns: scale out fast (60s), scale in cautiously (300s) to avoid thrashing.
autoscaling_config = {
    # Step 1: Register the endpoint as a scalable target
    'register': {
        'ServiceNamespace': 'sagemaker',
        'ResourceId': f'endpoint/{ENDPOINT_NAME}/variant/primary',
        'ScalableDimension': 'sagemaker:variant:DesiredInstanceCount',
        'MinCapacity': 1,
        'MaxCapacity': 10,
    },
    # Step 2: Create a scaling policy
    'policy': {
        'PolicyName': 'iris-invocations-scaling',
        'PolicyType': 'TargetTrackingScaling',
        'TargetTrackingScalingPolicyConfiguration': {
            'PredefinedMetricSpecification': {
                'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'
            },
            'TargetValue': 1000.0,     # scale out when > 1000 req/min per instance
            'ScaleInCooldown': 300,    # wait 5 min before scaling in
            'ScaleOutCooldown': 60,    # scale out quickly (1 min)
        }
    }
}

# Two API calls: register the scalable target, then attach the tracking policy.
try:
    aas = boto3.client('application-autoscaling', region_name='us-east-1')
    aas.register_scalable_target(**autoscaling_config['register'])

    policy_params = autoscaling_config['policy'].copy()
    policy_params['ServiceNamespace'] = 'sagemaker'
    policy_params['ResourceId'] = autoscaling_config['register']['ResourceId']
    policy_params['ScalableDimension'] = autoscaling_config['register']['ScalableDimension']
    aas.put_scaling_policy(**policy_params)
    print("Auto-scaling policy created.")

except NoCredentialsError:
    print("[No credentials] Would configure auto-scaling:")
    print(f"  Min instances : {autoscaling_config['register']['MinCapacity']}")
    print(f"  Max instances : {autoscaling_config['register']['MaxCapacity']}")
    print(f"  Scale trigger : > 1000 invocations/min/instance")
    print(f"  Scale-out wait: 60 seconds")
    print(f"  Scale-in wait : 300 seconds (conservative to avoid thrashing)")

except ClientError as e:
    print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")

[No credentials] Would configure auto-scaling:
  Min instances : 1
  Max instances : 10
  Scale trigger : > 1000 invocations/min/instance
  Scale-out wait: 60 seconds
  Scale-in wait : 300 seconds (conservative to avoid thrashing)


## 9. Delete the Endpoint to Avoid Charges

SageMaker endpoints bill by the hour even when idle. Always delete endpoints you are not using.

In [9]:
# WHAT: delete the endpoint (and show the full cleanup checklist).
# WHY: endpoints bill per hour whether or not they serve traffic — forgetting
# this step is the classic cloud-ML budget mistake.
try:
    sm = boto3.client('sagemaker', region_name='us-east-1')
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"Endpoint '{ENDPOINT_NAME}' deletion initiated.")
    print("Billing stops when the endpoint reaches 'Deleted' status.")

except NoCredentialsError:
    print("[No credentials] Would call: sm.delete_endpoint(EndpointName='iris-classifier-endpoint')")
    print()
    print("Cleanup checklist in production:")
    print("  1. sm.delete_endpoint(EndpointName=...)          # stop billing")
    print("  2. sm.delete_endpoint_config(EndpointConfigName=...) # clean up config")
    print("  3. sm.delete_model(ModelName=...)                # clean up model registry")
    print("  4. s3.delete_object(Bucket=..., Key=...)         # optional: remove artifact")

# 'ValidationException' here just means the endpoint never existed — that is fine.
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException':
        print("Endpoint not found (already deleted or never created).")
    else:
        print(f"[AWS error] {e.response['Error']['Code']}: {e.response['Error']['Message']}")

[No credentials] Would call: sm.delete_endpoint(EndpointName='iris-classifier-endpoint')

Cleanup checklist in production:
  1. sm.delete_endpoint(EndpointName=...)          # stop billing
  2. sm.delete_endpoint_config(EndpointConfigName=...) # clean up config
  3. sm.delete_model(ModelName=...)                # clean up model registry
  4. s3.delete_object(Bucket=..., Key=...)         # optional: remove artifact


## Summary

The SageMaker deployment flow has five steps:
1. **Package** — create `model.tar.gz` with your model artifact and inference script
2. **Upload** — push the artifact to an S3 bucket the SageMaker execution role can access
3. **Register** — call `create_model()` pointing to the S3 URI and container image
4. **Configure** — call `create_endpoint_config()` to specify instance type and count
5. **Deploy** — call `create_endpoint()` and wait for `InService` status

Key things to remember:
- The endpoint config is separate from the endpoint, enabling safe blue/green updates
- Auto-scaling uses `SageMakerVariantInvocationsPerInstance` as the default trigger
- Endpoints bill by the hour — delete them when not in use

## Self-Check

1. **What format does SageMaker expect the model artifact in?**
   *(Name the exact file format and what should be inside.)*

2. **What is an endpoint config and why is it separate from the endpoint?**
   *(What does separating them allow you to do when you update a model?)*

3. **How do you avoid being charged for an idle SageMaker endpoint?**
   *(One specific API call. What happens to billing when you make it?)*

## 📚 References

1. Sculley, D., Holt, G., Golovin, D., et al. (2015). *Hidden Technical Debt in Machine Learning Systems*. NeurIPS 28. <https://papers.nips.cc/paper_files/paper/2015/hash/86df7dcfd896fcaf2674f757a2463eba-Abstract.html>
2. Paleyes, A., Urma, R.-G., & Lawrence, N. D. (2022). *Challenges in Deploying Machine Learning: A Survey of Case Studies*. ACM Computing Surveys. <https://arxiv.org/abs/2011.09926>
3. Shankar, S., Garcia, R., Hellerstein, J. M., & Parameswaran, A. G. (2022). *Operationalizing Machine Learning: An Interview Study*. arXiv. <https://arxiv.org/abs/2209.09125>
